# Week 3 Exercise — Synthetic Data Generator

Create your own tool that generates **synthetic data / test data**. Input the type of dataset (e.g. products, job postings, customer records) and the tool uses an LLM to generate sample rows.

- **Input:** Description of the data you want + number of samples
- **Output:** Structured data (table view + raw JSON)
- **Model:** OpenAI only (no extra API keys)

In [ ]:
# Imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"OpenAI API key loaded (starts with {api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")

MODEL = "gpt-4.1-mini"
client = OpenAI() if api_key else None

In [ ]:
def generate_synthetic_data(description: str, num_samples: int) -> tuple[str, str]:
    """
    Ask the LLM to generate a JSON array of synthetic records.
    Returns (table_html_or_error, raw_json_str).
    """
    if not description or not description.strip():
        return "Please describe the type of data you want.", ""
    if not client:
        return "OpenAI client not initialized. Set OPENAI_API_KEY.", ""
    n = max(1, min(int(num_samples) if isinstance(num_samples, (int, float)) else 5, 50))
    system = (
        f"You are a synthetic data generator. Produce a JSON object with one key (e.g. 'data' or 'records') "
        f"whose value is an array of exactly {n} objects based on the user's description. "
        "Use consistent field names across all objects. Output ONLY valid JSON, no markdown or extra text."
    )
    raw = ""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": description.strip()},
            ],
            response_format={"type": "json_object"},
        )
        raw = resp.choices[0].message.content
        data = json.loads(raw)
        # Accept either {"data": [...]} or direct array
        if isinstance(data, list):
            rows = data
        elif isinstance(data, dict):
            for key in ("data", "items", "records", "rows"):
                if key in data and isinstance(data[key], list):
                    rows = data[key]
                    break
            else:
                rows = list(data.values())[0] if data else []
                if not isinstance(rows, list):
                    rows = [data]
        else:
            rows = [data]
        if not rows:
            return "No array of objects found in the response.", raw
        df = pd.DataFrame(rows)
        return df.to_html(index=False), raw
    except json.JSONDecodeError as e:
        return f"JSON parse error: {e}", raw
    except Exception as e:
        return f"Error: {e}", ""

In [ ]:
def run_ui(description: str, num_samples: int):
    table, raw = generate_synthetic_data(description, num_samples)
    return table, raw

with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("### Generate synthetic / test data")
    desc = gr.Textbox(
        label="Describe the data",
        placeholder="e.g. 10 e-commerce products with name, price, category, sku",
        lines=3,
    )
    num = gr.Number(label="Number of samples", value=5, minimum=1, maximum=50, step=1)
    btn = gr.Button("Generate")
    table_out = gr.HTML(label="Table")
    json_out = gr.Code(label="Raw JSON", language="json")
    btn.click(fn=run_ui, inputs=[desc, num], outputs=[table_out, json_out])
    gr.Examples(
        [
            ["5 job postings for software engineers in NYC with title, company, salary range, and requirements", 5],
            ["8 customer support tickets with id, subject, priority, status, created_at", 8],
            ["10 product listings: name, price, category, in_stock", 10],
        ],
        inputs=[desc, num],
        label="Examples",
    )

demo.launch()